# Tensor Shapes, Memory & Autograd for LLM

**Roadmap 1.1 / scope:** active recall L2, exploratory notebook for future L3 practice. No external model or dataset. Requires PyTorch 2.x; runs on CPU. Saved output text corresponds to these deterministic examples. Execute top-to-bottom.

**Question before each cell:** predict shape, strides, aliasing or gradient first, then compare.

In [1]:
import torch
print("Runtime: CPU; torch imported")

Runtime: CPU; torch imported


## 1. A scalar value is not necessarily a scalar **tensor**

The number of stored values (`numel`) and the number of axes (`ndim`) are different properties.

In [2]:
items = {
    "scalar": torch.tensor(5.0),
    "vector": torch.tensor([5.0]),
    "matrix": torch.tensor([[5.0]]),
}
for name, t in items.items():
    print(name, "shape=", tuple(t.shape), "ndim=", t.ndim, "numel=", t.numel)

scalar shape= () ndim= 0 numel= 1
vector shape= (1,) ndim= 1 numel= 1
matrix shape= (1, 1) ndim= 2 numel= 1


## 2. Axes and indexing in token embeddings

Batch × Tokens × Features. Integer indices **remove** axes; slicing keeps them. Use float dtype so reduction by `mean` is valid.

In [3]:
x = torch.arange(24, dtype=torch.float32).reshape(2, 3, 4)
print("X:", tuple(x.shape), "ndim:", x.ndim, "numel:", x.numel())
print("X[1]:", tuple(x[1].shape))
print("X[1:2]:", tuple(x[1:2].shape))
print("X[:,2]:", tuple(x[:, 2].shape))
print("X.mean(dim=1):", tuple(x.mean(dim=1).shape))
print("X[1,2,3]:", x[1, 2, 3].item())

X: (2, 3, 4) ndim: 3 numel: 24
X[1]: (3, 4)
X[1:2]: (1, 3, 4)
X[:,2]: (2, 4)
X.mean(dim=1): (2, 4)
X[1,2,3]: 23.0


## 3. Dot vs outer vs cosine

`a @ b` returns a scalar, while `torch.outer(a,b)` returns an all-pairs matrix.

In [4]:
a = torch.tensor([1., 2.])
b = 2 * a
print("dot:", (a @ b).item(), "shape:", tuple((a @ b).shape))
print("outer:", torch.outer(a, b).tolist())
print("cosine:", torch.nn.functional.cosine_similarity(a, b, dim=0).item())

dot: 10.0 shape: ()
outer: [[2.0, 4.0], [4.0, 8.0]]
cosine: 0.9999999403953552


## 4. `permute` changes the interpretation of data, not physical storage

For a contiguous tensor (2,3,4), strides are (12,4,1). After `permute(0,2,1)`, strides become (12,1,4).

In [5]:
y = x.permute(0, 2, 1)
print("X shape/stride:", tuple(x.shape), x.stride())
print("Y shape/stride:", tuple(y.shape), y.stride())
print("Y[1,2,0] == X[1,0,2]:", y[1, 2, 0].item(), x[1, 0, 2].item())
print("Y contiguous:", y.is_contiguous())
assert y[1, 2, 0] == x[1, 0, 2]

X shape/stride: (2, 3, 4) (12, 4, 1)
Y shape/stride: (2, 4, 3) (12, 1, 4)
Y[1,2,0] == X[1,0,2]: 14.0 14.0
Y contiguous: False


## 5. `view` vs `reshape`

For the transposed 2D example, `view(12)` cannot represent the desired logical order without a copy. `reshape` may copy. The pointer check below is **for this example**, not a promise about all reshapes.

In [6]:
transposed = torch.arange(12).reshape(3, 4).T
try:
    transposed.view(12)
    print("view worked")
except RuntimeError:
    print("view incompatible with strides")
reshaped = transposed.reshape(12)
print("reshape created new storage:", reshaped.untyped_storage().data_ptr() != transposed.untyped_storage().data_ptr())

view incompatible with strides
reshape created new storage: True


## 6. Broadcasting without materializing repeats

`expand` adds zero strides on broadcasted axes; `repeat` allocates a new tensor. Compare only raw data bytes; metadata is not counted.

In [7]:
bias16 = torch.tensor([[[1., 2., 3., 4.]]], dtype=torch.float16)
expanded = bias16.expand(2, 3, 4)
repeated = bias16.repeat(2, 3, 1)
print("expand shape/stride:", tuple(expanded.shape), expanded.stride())
print("base data bytes:", bias16.numel() * bias16.element_size())
print("repeat new bytes:", repeated.numel() * repeated.element_size())

expand shape/stride: (2, 3, 4) (0, 0, 1)
base data bytes: 8
repeat new bytes: 48


## 7. Parameter sharing and autograd

A scalar loss gives a **tensor of gradients per parameter**. Six (2×3) token positions use each of four bias features. Compare `sum` vs `mean` loss.

In [8]:
embeddings = torch.zeros(2, 3, 4)
learnable_bias = torch.tensor([[[1., 2., 3., 4.]]], requires_grad=True)
(embeddings + learnable_bias).sum().backward()
print("sum grad:", learnable_bias.grad.flatten().tolist())
learnable_bias.grad.zero_()
(embeddings + learnable_bias).mean().backward()
print("mean grad:", learnable_bias.grad.flatten().tolist())
print("grad shape:", tuple(learnable_bias.grad.shape))

sum grad: [6.0, 6.0, 6.0, 6.0]
mean grad: [0.25, 0.25, 0.25, 0.25]
grad shape: (1, 1, 4)


## Active recall and next steps

Predict before running: (1) reshape a (B,T,H,Dh) tensor into (B,H,T,Dh) without permute: does it preserve head/token meanings? (2) Why is a scalar loss not a scalar gradient? (3) When would `.contiguous()` copy?

**Evidence boundary:** These cells are CPU teaching demos, not independent L3 GPU profiling evidence or performance benchmarks. Source notes: [Tensor Fundamentals](../notes/01-tensor-fundamentals-for-llm.md).